### Setup

In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import (apriori, association_rules,fpgrowth)
import time

In [2]:
games_df = pd.read_csv("gameDim.csv",encoding="latin1")
fact_table=pd.read_csv("steamUserFact.csv")

In [3]:
fact_table=fact_table.drop_duplicates()

In [ ]:
games_df.head()

In [5]:
games_df=games_df.drop(columns=["specs","release_date","price","genres","tags"])

In [6]:
fact_table=fact_table[fact_table["playtime_hours"]>2]

remove games which were rarely played by user

In [7]:
final = pd.merge(fact_table,games_df,on="game_id",how="left")

In [8]:
transactions=[]
for i,j in final.groupby("steam_id"):
    transactions.append(list(j["game_name"]))

get all transaction occured

In [9]:
transactions[:2]

[['Automobilista',
  'F1 2014',
  'F1 2015',
  'GRID Autosport',
  'Burnout Paradise: The Ultimate Box',
  'Game Dev Tycoon',
  'Cities: Skylines',
  'Counter-Strike',
  'SMITE®',
  'Insurgency',
  'Far Cry 3',
  'Fallout: New Vegas',
  'Fallout 4',
  'Grand Theft Auto V',
  'DayZ',
  'Saints Row: The Third',
  'PlanetSide 2',
  'Counter-Strike: Source'],
 ['SpellForce 2 - Anniversary Edition',
  'SpellForce - Platinum Edition',
  'Gnomoria',
  "Recettear: An Item Shop's Tale",
  'Farming Simulator 2013 Titanium Edition',
  'Europa Universalis IV',
  'Crusader Kings II',
  'Factorio',
  "No Man's Sky",
  'DARK SOULS\x99 II',
  'DARK SOULS\x99 III',
  'Warhammer 40,000: Dawn of War II',
  'The Walking Dead',
  'Kerbal Space Program',
  'FTL: Faster Than Light',
  'Torchlight II',
  'Grand Theft Auto V',
  'Rocket League®',
  'Counter-Strike: Source',
  'Borderlands 2',
  'Terraria']]

In [10]:
transaction_enc= TransactionEncoder()
encoded = transaction_enc.fit_transform(transactions)

use transaction enocder as it is required by mlxtend

In [11]:
bask = pd.DataFrame(encoded,columns=transaction_enc.columns_)

In [12]:
bask.head()

,"""Glow Ball"" - The billiard puzzle game",//N.P.P.D. RUSH//- The milk of Ultraviolet,//SNOWFLAKE TATTOO//,001 Game Creator,0RBITALIS,10 Second Ninja,10 Second Ninja X,10 Years After,"10,000,000",100% Orange Juice,...,rFactor,rFactor 2,realMYST,realMyst: Masterpiece Edition,resident evil 4 / biohazard 4,rymdkapsel,sZone-Online,the static speaks my name,theHunter Classic,theHunter: Primal
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Apriori

In [13]:
start=time.time()
freq_item_set= apriori(bask,min_support=0.02,use_colnames=True)
end=time.time()

In [14]:
end-start

3.8266608715057373

In [15]:
freq_item_set.sample(5)

,support,itemsets
639,0.027520,"frozenset({Fallout 4, Terraria, Borderlands 2})"
170,0.032052,"frozenset({ARK: Survival Evolved, Unturned})"
419,0.032693,"frozenset({Warframe, Grand Theft Auto V})"
608,0.051618,"frozenset({Terraria, Trove})"
379,0.030927,"frozenset({Fallout 4, Portal 2})"


In [16]:
rules_apriori = association_rules(freq_item_set,metric='lift',min_threshold=1)[["antecedents","consequents","support","confidence","lift"]]

In [17]:
rules_apriori.sort_values(by="lift",ascending=False).head(20)

,antecedents,consequents,support,confidence,lift
489,frozenset({Half-Life 2: Episode One}),frozenset({Half-Life 2: Episode Two}),0.020128,0.668050,21.427488
488,frozenset({Half-Life 2: Episode Two}),frozenset({Half-Life 2: Episode One}),0.020128,0.645614,21.427488
475,frozenset({Half-Life 2: Episode One}),frozenset({Half-Life 2}),0.025332,0.840768,7.295888
474,frozenset({Half-Life 2}),frozenset({Half-Life 2: Episode One}),0.025332,0.219826,7.295888
476,frozenset({Half-Life 2: Episode Two}),frozenset({Half-Life 2}),0.025817,0.828070,7.185704
477,frozenset({Half-Life 2}),frozenset({Half-Life 2: Episode Two}),0.025817,0.224030,7.185704
1705,frozenset({The Binding of Isaac: Rebirth}),"frozenset({Terraria, The Binding of Isaac})",0.020707,0.335953,6.514339
1700,"frozenset({Terraria, The Binding of Isaac})",frozenset({The Binding of Isaac: Rebirth}),0.020707,0.401515,6.514339
843,frozenset({Arma 3}),"frozenset({DayZ, Arma 2: Operation Arrowhead})",0.027270,0.291952,6.419845
842,"frozenset({DayZ, Arma 2: Operation Arrowhead})",frozenset({Arma 3}),0.027270,0.599656,6.419845


we can see it working as episode one of a game and episode two are recommended together

### fp growth

In [18]:
fp_itemset = fpgrowth(bask,min_support=0.02,use_colnames=True)

In [19]:
rules_fp = association_rules(fp_itemset,metric='lift',min_threshold=1)[["antecedents","consequents","support","confidence","lift"]]

In [20]:
rules_fp.sort_values(by="lift",ascending=False).head(20)

,antecedents,consequents,support,confidence,lift
724,frozenset({Half-Life 2: Episode Two}),frozenset({Half-Life 2: Episode One}),0.020128,0.645614,21.427488
725,frozenset({Half-Life 2: Episode One}),frozenset({Half-Life 2: Episode Two}),0.020128,0.668050,21.427488
722,frozenset({Half-Life 2}),frozenset({Half-Life 2: Episode One}),0.025332,0.219826,7.295888
723,frozenset({Half-Life 2: Episode One}),frozenset({Half-Life 2}),0.025332,0.840768,7.295888
720,frozenset({Half-Life 2: Episode Two}),frozenset({Half-Life 2}),0.025817,0.828070,7.185704
721,frozenset({Half-Life 2}),frozenset({Half-Life 2: Episode Two}),0.025817,0.224030,7.185704
1075,frozenset({The Binding of Isaac: Rebirth}),"frozenset({Terraria, The Binding of Isaac})",0.020707,0.335953,6.514339
1070,"frozenset({Terraria, The Binding of Isaac})",frozenset({The Binding of Isaac: Rebirth}),0.020707,0.401515,6.514339
1115,frozenset({Arma 3}),"frozenset({DayZ, Arma 2: Operation Arrowhead})",0.027270,0.291952,6.419845
1114,"frozenset({DayZ, Arma 2: Operation Arrowhead})",frozenset({Arma 3}),0.027270,0.599656,6.419845


same results as apriori

### comparison

In [21]:
def ev(freq_i,name):
    start=time.time()
    rules = association_rules(freq_i,metric='lift',min_threshold=1)[["antecedents","consequents","support","confidence","lift"]]
    end=time.time()
    return {"model":name,"Rules Length":len(rules),"Time Taken":end-start,"Average Support":rules["support"].mean(),"Average Confidence":rules["confidence"].mean(),"Average Lift":rules["lift"].mean()}

In [22]:
apriori = ev(freq_item_set,"Apriori")
fp = ev(fp_itemset,"FP_Growth")

In [23]:
pd.DataFrame([apriori,fp])

,model,Rules Length,Time Taken,Average Support,Average Confidence,Average Lift
0,Apriori,1730,0.007385,0.030842,0.265218,1.745984
1,FP_Growth,1730,0.006583,0.030842,0.265218,1.745984


only time was different with fp growth being faster

### Recommendation system

In [92]:
def recommend(games,rules):
    return rules[[antecedent.issubset(games) for antecedent in rules["antecedents"]]].sort_values(["lift"],ascending=False).head(2)

In [93]:
my_games=["Terraria","PlanetSide 2"]
recommend(my_games,rules_fp)

,antecedents,consequents,support,confidence,lift
1352,frozenset({Terraria}),"frozenset({Starbound, Portal 2})",0.025286,0.072071,2.267339
1560,frozenset({PlanetSide 2}),frozenset({Blacklight: Retribution}),0.026895,0.162804,2.253929


recommend games based on